# Lesson 00 — Colab Setup برای AI Workspace

**هدف:** محیط Colab را برای بقیهٔ مسیر آماده کنی (GPU، پکیج‌ها، اتصال به ریپو).

**محصول نهایی:** AI Workspace شبیه Notion + Browser Agent.

---

## چرا این درس؟
بدون GPU و نسخهٔ درست کتابخانه‌ها، LoRA و RAG بعدی یا کند می‌شود یا می‌ترکد.

## نحوهٔ باز کردن در Colab
1. این فایل `.ipynb` را آپلود کن یا از ریپو باز کن.
2. Runtime → Change runtime type → **GPU (T4)**.
3. همهٔ cellها را به ترتیب Run کن.

## 1) چک GPU و نسخهٔ Python

In [1]:
import sys
print("Python:", sys.version)

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print("WARNING: No GPU. Runtime → Change runtime type → GPU, then Restart session.")

Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM (GB): 15.64


## 2) نصب پکیج‌های مسیر یادگیری

In [2]:
%pip install -q "transformers>=4.44" "datasets" "accelerate" "peft" "bitsandbytes" \
  "sentence-transformers" "chromadb" "fastapi" "uvicorn" "httpx" "pydantic" "einops"

print("Install done.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 128.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/6

## 3) اتصال Google Drive (توصیه‌شده)

checkpointهای LoRA و دیتاست را روی Drive نگه دار.

In [4]:
from pathlib import Path

USE_DRIVE = True  # اگر Drive نمی‌خواهی False کن

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/ai-workspace-colab")
else:
    ROOT = Path("/content/ai-workspace-colab")

ROOT.mkdir(parents=True, exist_ok=True)
(ROOT / "datasets").mkdir(exist_ok=True)
(ROOT / "checkpoints").mkdir(exist_ok=True)
(ROOT / "evals").mkdir(exist_ok=True)

print("Work dir:", ROOT)

Mounted at /content/drive
Work dir: /content/drive/MyDrive/ai-workspace-colab


## 4) کلون ریپو (اختیاری — وقتی روی GitHub بود)

In [7]:
REPO_URL = "https://github.com/erfanhamidi9574-dot/ai-test.git"  # مثال: "https://github.com/USER/ai-workspace.git"

from pathlib import Path
import subprocess

repo_dir = Path("/content/ai-workspace")
if REPO_URL:
    if repo_dir.exists():
        subprocess.run(["git", "-C", str(repo_dir), "pull"], check=False)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
    print("Repo at:", repo_dir)
else:
    print("REPO_URL empty — ok for Lesson 00.")

Repo at: /content/ai-workspace


## 5) تست tokenizer (بدون بارگذاری کامل مدل)

In [10]:
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

messages = [
    {"role": "system", "content": "You are an AI assistant inside a Notion-like workspace."},
    {"role": "user", "content": "Rewrite this block to be clearer: Ship AI workspace mvp soon"},
]
text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(text[:500])
print("\nTokenizer OK — ready for Lesson 01")

<|im_start|>system
You are an AI assistant inside a Notion-like workspace.<|im_end|>
<|im_start|>user
Rewrite this block to be clearer: Ship AI workspace mvp soon<|im_end|>
<|im_start|>assistant


Tokenizer OK — ready for Lesson 01


## تمرین‌های Lesson 00

1. GPU را روشن کن تا `CUDA available: True` ببینی.
2. مسیر `ROOT` را یادداشت کن.
3. هدف محصول را در cell بعدی در ۲ جمله بنویس.

وقتی تمام شد در چت بگو: **درس 00 تمام**

### چک‌لیست
- [ ] GPU فعال
- [ ] پکیج‌ها نصب شدند
- [ ] پوشهٔ کار ساخته شد
- [ ] tokenizer تست شد

In [ ]:
MY_PRODUCT_GOAL = """
(اینجا بنویس)
"""
print(MY_PRODUCT_GOAL.strip() or "WARNING: still empty")